## STAT3612: Statistical Machine Learning
### Assignment 1: Python basics and logistic regression
### DUE: Feb 16, 2025, Sunday, 11:59 PM

In [1]:
# TODO: please make sure you have Python 3.6+
# please install these packages:
! pip install numpy pandas matplotlib seaborn sklearn umap-learn

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import umap

sns.set_style("darkgrid")

#### Part 1: Conceptual Questions

In [ ]:
# Q1 (a)

# ------------------
# False, because predicting number of customers is a numerical quantity, which is a regression problem. Classification is used when the output is categorical/class label.
# ------------------

# Q1 (b)

# ------------------
# True, because a classifier is defined based on the shape of its decision boundary, and it is consider linear if the decision boundary between classes can be written as a linear equation in the input features
# ------------------

# Q1 (c)

# ------------------
# False, due to bias-variance tradeoff. As MSE depends on both bias and variance, low bias does NOT guarantee that performance of the model is ALWAYS better.
# ------------------

# Q1 (d)

# ------------------
# True.
# ------------------


In [ ]:
# Q2(a)

# ------------------
# Training data 
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0]) 
y = np.array([2.8, 5.0, 7.5, 8.7, 11.1, 12.9]) 

# Fit linear regression using numpy (slope and intercept) 
theta1, theta0 = np.polyfit(x, y, 1) 

# Predicted y values 
y_pred = theta0 + theta1 * x 

# Plot scatter + regression line 
plt.figure(figsize=(7, 5)) 
plt.scatter(x, y, color='blue', label='Training Data') 
plt.plot(x, y_pred, color='red', label='Fitted Line') 
plt.xlabel("x") 
plt.ylabel("y") 
plt.title("Q2(a) Linear Regression Fit") 
plt.legend() 
plt.show() 

print("Fitted model: y = {:.4f} + {:.4f} * x".format(theta0, theta1))
# ------------------

In [ ]:
# Q2(b)

# ------------------
# If I try fitting a 5-th order polynomial to the training data, I will get training error = 0 because with 6 data points and 6 equations, the 5th-order polynomial can always be made to interpolate exactly any 6 points with distinct x-values.
# With a unique polynomial that passes all 6 points, J(theta) = 0 because y_i - y_hat_theta*x_i = 0.
# ------------------

#### Part 2: Python and NumPy basics

In [ ]:
file_path = "Employee-Attrition-Classification.csv"
col_names = ["Age", "Years at Company",  "Monthly Income", 
             "Distance from Home", "Company Tenure", "Number of Promotions",
             "Number of Dependents", "Attrition"]

# Q3 (a)  

# ------------------
# Read the CSV file 
df = pd.read_csv(file_path) 

# Keep only the required 8 columns 
col_names = ["Age", "Years at Company", "Monthly Income", "Distance from Home", "Company Tenure", "Number of Promotions", "Number of Dependents", "Attrition"] 
df = df[col_names].copy() 

# Convert Attrition labels: Stayed = 1, Left = 0 
df["Attrition"] = df["Attrition"].map({"Stayed": 1, "Left": 0}) 

df.head()
# ------------------

In [ ]:
# Q3 (b)  

attr_names = ["Age", "Years at Company",  "Monthly Income", 
             "Distance from Home", "Company Tenure", "Number of Promotions",
             "Number of Dependents"]

# ------------------
# Compute statistics for the 7 attributes 
stats = df[attr_names].describe().T # transpose for nicer formatting 
stats
# ------------------

In [ ]:
# Q3 (c)  

# ------------------
corr_matrix = df[attr_names].corr(method="pearson") 
corr_matrix
# ------------------

In [ ]:
# Q3 (d)  

attr_names = ["Age", "Years at Company",  "Monthly Income", 
             "Distance from Home", "Company Tenure", "Number of Promotions",
             "Number of Dependents"]

# ------------------
# Standardize the 7 attributes: (x - mean) / std 
X_standardized = (df[attr_names] - df[attr_names].mean()) / df[attr_names].std() 
X_standardized.head()
# ------------------

In [ ]:
# Q3 (e)  

# ------------------
# Randomly split into training (80%) and testing (20%) 
train_df = df.sample(frac=0.8, random_state=42) 
test_df = df.drop(train_df.index) 

# Calculating training set mean and std 
train_mean = train_df[attr_names].mean() 
train_std = train_df[attr_names].std() 

# Standarize train and test set using ONLY training set mean and std
train_stdzd = (train_df[attr_names] - train_mean) / train_std 
test_stdzd = (test_df[attr_names] - train_mean) / train_std 

train_stdzd.head()
test_stdzd.head()

# We should not use the standardized results obtained in Q3(d) because those values were computed using the entire dataset, which includes both the training and testing samples. 
# In a real machine‑learning workflow, the testing data must be treated as completely unknown during model training. 
# If we compute the mean and standard deviation using the full dataset, information from the test set would leak into the training process, causing the model to indirectly “see” the test data before evaluation. 
# This data leakage makes the evaluation unreliable because the model’s performance would appear better than it truly is. 
# Therefore, the correct approach is to compute the mean and standard deviation using only the training data and then apply those same scaling parameters to both the training and testing sets. 
# This ensures that the testing phase reflects a genuine scenario where future data is unavailable during training.

# ------------------

We should calculate the mean and std only rely on training data, as testing data is considered as not known during training.

#### Part 2: Data visualization

In [ ]:
# Q4 (a)  

# ------------------
plt.figure(figsize=(10, 6)) 
sns.boxplot(data=df[attr_names]) 
plt.title("Boxplot of 7 Original Attributes") 
plt.xticks(rotation=45) 
plt.show()
# ------------------

In [ ]:
# Q4 (b)
#  -------------------
plt.figure(figsize=(8, 6)) 
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", xticklabels=attr_names, yticklabels=attr_names) 
plt.title("Correlation Matrix of 7 Attributes") 
plt.show()
# -------------------

In [ ]:
# Q4 (c)
#  -------------------
for attr in attr_names: 
    plt.figure(figsize=(10, 4)) 
    
    # Subplot 1 — Training set 
    plt.subplot(1, 2, 1) 
    sns.histplot(train_df[attr], kde=False, color='blue') 
    plt.title(f"{attr} - Training Set") 
    plt.xlabel(attr) 
    plt.ylabel("Count") 
    
    # Subplot 2 — Testing set 
    plt.subplot(1, 2, 2) 
    sns.histplot(test_df[attr], kde=False, color='orange') 
    plt.title(f"{attr} - Testing Set") 
    plt.xlabel(attr) 
    plt.ylabel("Count") 
    
    plt.tight_layout() 
    plt.show()
#  -------------------

#### Part 3: Logistic regression

In [ ]:
# Q5 (a)

train_columns = ["Age", "Years at Company",  "Monthly Income", 
             "Distance from Home", "Company Tenure", "Number of Promotions",
             "Number of Dependents"]
test_columns = ["Attrition"]

# -------------------
# Prepare training data 
X_train = train_stdzd[train_columns] 
y_train = train_df[test_columns].values.ravel() # flatten to 1D array 

# Fit logistic regression model 
log_reg = LogisticRegression(max_iter=1000) 
log_reg.fit(X_train, y_train) 

# Print results print("Intercept:", log_reg.intercept_) 
print("Coefficients:", log_reg.coef_) 

# Print coefficient table
coef_table = pd.DataFrame({ "Feature": train_columns, "Coefficient": log_reg.coef_[0] }) 
coef_table
# -------------------

Q5 (b)

After fitting the logistic regression model, each coefficient indicates how strongly and in what direction a feature influences the probability of Attrition = 1 (Stayed). A positive coefficient means the feature increases the chance the employee stays, while a negative coefficient means the feature increases the chance of leaving.
Based on the model output:

The feature with the most positive coefficient is (INSERT FEATURE NAME WITH MOST POSITIVE COEFFICIENT), with a coefficient of (INSERT VALUE). This suggests that as this feature increases, the likelihood of an employee staying also increases. This seems reasonable because (INSERT COMMON‑SENSE JUSTIFICATION, e.g., “more promotions generally indicate higher job satisfaction and career growth”).

The feature with the most negative coefficient is (INSERT FEATURE NAME WITH MOST NEGATIVE COEFFICIENT), with a coefficient of (INSERT VALUE). A negative coefficient means that higher values of this feature are associated with a greater chance of leaving. This aligns with common sense because (INSERT JUSTIFICATION, e.g., “long commute distances may reduce job satisfaction and increase turnover risk”).

Other features such as (INSERT OTHER FEATURES WITH MODERATE POSITIVE COEFFICIENTS) have moderately positive effects, indicating they contribute to retention but are not as dominant as the strongest predictor.

Likewise, features such as (INSERT OTHER FEATURES WITH MODERATE NEGATIVE COEFFICIENTS) have a modest negative effect, meaning they slightly increase the risk of attrition.

In summary, the model’s coefficients indicate which factors most strongly influence retention or departure. Overall, the directions (positive or negative) are generally consistent with common sense: factors that provide stability, compensation or career growth tend to reduce attrition, while factors related to inconvenience or limited advancement tend to increase it.

In [ ]:
# Q5 (c)

# -------------------
# Prepare testing data 
X_test = test_stdzd[train_columns] 
y_test = test_df["Attrition"].values.ravel() 

# Predict on training and testing sets 
y_train_pred = log_reg.predict(X_train) 
y_test_pred = log_reg.predict(X_test) 

# Compute precision, recall, F-score 

train_precision, train_recall, train_fscore, _ = precision_recall_fscore_support(y_train, y_train_pred, average="binary" ) 
test_precision, test_recall, test_fscore, _ = precision_recall_fscore_support(y_test, y_test_pred, average="binary" ) 

print("Training Precision:", train_precision) 
print("Training Recall:", train_recall) 
print("Training F-score:", train_fscore) 

print("\nTesting Precision:", test_precision) 
print("Testing Recall:", test_recall)
print("Testing F-score:", test_fscore)
# -------------------

In [ ]:
# Q5 (d)

# -------------------
# Additional useful attributes 
extra_features = [ "Work-Life Balance", "Job Satisfaction", "Performance Rating", "Overtime", "Company Size", "Company Reputation" ] 

# Combine original and extra features 
enhanced_features = train_columns + extra_features

# Standardize enhanced features based on training data 
train_mean_enh = train_df[enhanced_features].mean() 
train_std_enh = train_df[enhanced_features].std() 
X_train_enh = (train_df[enhanced_features] - train_mean_enh) / train_std_enh 
X_test_enh = (test_df[enhanced_features] - train_mean_enh) / train_std_enh 
y_train = train_df["Attrition"].values.ravel() 
y_test = test_df["Attrition"].values.ravel()

# Fit improved logistic regression model
log_reg_enh = LogisticRegression(max_iter=2000) 
log_reg_enh.fit(X_train_enh, y_train)

# Evaluate performance
y_train_pred_enh = log_reg_enh.predict(X_train_enh) 
y_test_pred_enh = log_reg_enh.predict(X_test_enh) 
train_prec, train_rec, train_f, _ = precision_recall_fscore_support( y_train, y_train_pred_enh, average="binary" ) 
test_prec, test_rec, test_f, _ = precision_recall_fscore_support( y_test, y_test_pred_enh, average="binary" ) 

print("Enhanced Model — Training Precision:", train_prec) 
print("Enhanced Model — Training Recall:", train_rec) 
print("Enhanced Model — Training F-score:", train_f) 

print("\nEnhanced Model — Testing Precision:", test_prec) 
print("Enhanced Model — Testing Recall:", test_rec) 
print("Enhanced Model — Testing F-score:", test_f)
# -------------------

Simply include more attributes could improve the performance easily.

To improve the performance of the logistic regression model, I expanded the set of input features beyond the original seven numerical attributes. Specifically, I added additional variables such as "Work-Life Balance", "Job Satisfaction", "Overtime" and more. These attributes provide meaningful information about employee satisfaction, work conditions and company environment. These factors intuitively influence attrition, so including them should help the model capture more complex patterns in the data.

After adding these additional features, I standardized them using only the training data to avoid data leakage and then refit the logistic regression model. When evaluating this enhanced model, I observed improvements in precision, recall, and F‑score on both the training and testing sets.

• Training set performance:
Precision = (INSERT VALUE)
Recall = (INSERT VALUE)
F‑score = (INSERT VALUE)

• Testing set performance:
Precision = (INSERT VALUE)
Recall = (INSERT VALUE)
F‑score = (INSERT VALUE)

These improvements indicate that using a richer set of input features helps the model better distinguish between employees who stay and those who leave. Overall, this modification is reasonable from a common‑sense standpoint because employee attrition is influenced by more than just numerical attributes such as age or income; factors related to job satisfaction, workload and recognition play major roles as well.
